In [88]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

In [89]:
CSV_PATH = r"D:\lixingjie\clean_data.csv"   
OUTPUT_FILE = r"D:\lixingjie\WPS_Multidimensional_Report.xlsx"

df = pd.read_csv(CSV_PATH, low_memory=False)
print(f"原始形状: {df.shape}")

原始形状: (153016, 113)


In [90]:
field_keywords = {
    'Project ID': 'Project ID',
    'Contractor': 'Contractor',
    'Unit': 'Unit',
    'Area Code': 'Area Code',
    'Area Name': 'Area Name',
    'Line No': 'Line No',
    'Joint No': 'Joint No',
    'Serv': 'Serv',
    'Pipe Class': 'Pipe Class',
    'WPS NO': 'WPS NO',
    'Position': 'Position',
    'Inch': 'Inch',
    'THK Code': 'THK.Code',
    'Joint Type': 'Joint Type',
    'Material': 'Material',
    'Diameter': 'Diameter',
    'THK': 'THK.',
    'Material Code': 'Material Code',
    'Inspection Code': 'Inspection Code',
    'NDT(%)': 'NDT(%)',
    'Welding Date': 'Welding Date',
    'Process': 'Process',
    'Fixed/Rotat': 'Fixed/Rotat',
    'Constractor teams': 'Constractor teams',
    'Root Welder': 'Root Welder',
    'Cover Welder': 'Cover Welder',
    'FitUp Date': 'FitUp Date',
    'Visual': 'Visual',
    'Final Report Date': 'Final Report Date',
    'PWHT': 'PWHT',
    'Test Package No': 'Test Package No',
    'RT State': 'RT State',
    'PT State': 'PT State',
    'UT State': 'UT State',
    'MT State': 'MT State',
    'NDT Films': 'NDT Films',
    'ACC Films': 'ACC. Films',
    'RT Date': 'RT Date',
    'PT Date': 'PT Date',
    'UT Date': 'UT Date',
    'MT Date': 'MT Date',
    'Repair Film No': 'Repair Film No',
    'Repair Welder': 'Repair Welder',
    'Repair Date': 'Repair Date',
    'repair State': 'repair State',
    'Re-repair Film No': 'Re-repair Film No.',
    'Re-repair Welder': 'Re-repair Welder',
    'Re-repair Date': 'Re-repair Date',
    'Re-repair State': 'Re-repair State',
    'data entry clerk': 'data entry clerk',
    'auditor': 'auditor',
    'date created': 'date created',
    'remark': 'remark'
}

col_map = {}
for std_name, keyword in field_keywords.items():
    matched = [col for col in df.columns if keyword in col]
    if len(matched) == 1:
        col_map[std_name] = matched[0]
    elif len(matched) > 1:
        col_map[std_name] = matched[0]
    else:
        print(f"未找到: {keyword}")

print(f"匹配成功: {len(col_map)} 个字段")

匹配成功: 53 个字段


In [91]:
df = df[[col for col in col_map.values()]]
df.rename(columns={v: k for k, v in col_map.items()}, inplace=True)

df.dropna(how='all', inplace=True)
print(f"清洗后: {df.shape[0]} 行, {df.shape[1]} 列")

清洗后: 153016 行, 53 列


In [92]:
cat_cols = ['Contractor', 'Unit', 'Area Code', 'Area Name', 'Line No', 'Serv',
            'Pipe Class', 'WPS NO', 'Position', 'THK Code', 'Joint Type',
            'Material', 'Material Code', 'Inspection Code', 'Process',
            'Fixed/Rotat', 'Constractor teams', 'Root Welder', 'Cover Welder',
            'Visual', 'PWHT', 'RT State', 'PT State', 'UT State', 'MT State',
            'repair State', 'Re-repair State', 'Test Package No',
            'data entry clerk', 'auditor']
for c in cat_cols:
    if c in df.columns:
        df[c] = df[c].astype('category')

date_cols = ['Welding Date', 'FitUp Date', 'Final Report Date',
             'RT Date', 'PT Date', 'UT Date', 'MT Date',
             'Repair Date', 'Re-repair Date', 'date created']
for c in date_cols:
    if c in df.columns:
        df[c] = pd.to_datetime(df[c], errors='coerce')

if 'NDT(%)' in df.columns:
    df['NDT(%)'] = pd.to_numeric(df['NDT(%)'], errors='coerce')
if 'NDT Films' in df.columns:
    df['NDT Films'] = pd.to_numeric(df['NDT Films'], errors='coerce')
if 'ACC Films' in df.columns:
    df['ACC Films'] = pd.to_numeric(df['ACC Films'], errors='coerce')

print(f"内存: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

内存: 61.46 MB


In [93]:
def norm_status(s):
    s = s.astype(str).str.strip().str.lower()
    return s.replace({
        '合格': '合格', 'acc': '合格', 'acceptable': '合格',
        '不合格': '不合格', 'reject': '不合格', 'fail': '不合格',
        '检测中': '检测中', 'testing': '检测中',
        'nan': '未检测', '': '未检测', 'none': '未检测'
    })

for c in ['RT State', 'PT State', 'UT State', 'MT State', 'Visual']:
    if c in df.columns:
        df[c] = norm_status(df[c])

if 'Welding Date' in df.columns:
    df['Welding_YearMonth'] = df['Welding Date'].dt.to_period('M')

In [94]:
print("开始多维分析...")
with pd.ExcelWriter(OUTPUT_FILE, engine='xlsxwriter') as writer:
    workbook = writer.book

    wps_main.to_excel(writer, sheet_name='WPS主维表', index=False)

    wps_mat = df.groupby(['WPS NO', 'Material']).agg(焊口数=('Joint No', 'nunique')).reset_index()
    wps_mat_pivot = wps_mat.pivot_table(index='WPS NO', columns='Material', values='焊口数', fill_value=0)
    wps_mat_pivot.to_excel(writer, sheet_name='WPS_材质交叉')

    if 'Constractor teams' in df.columns:
        wps_team = df.groupby(['WPS NO', 'Constractor teams']).agg(焊口数=('Joint No', 'nunique')).reset_index()
        wps_team_pivot = wps_team.pivot_table(index='WPS NO', columns='Constractor teams', values='焊口数', fill_value=0)
        wps_team_pivot.to_excel(writer, sheet_name='WPS_班组交叉')

    if 'Welding_YearMonth' in df.columns:
        monthly = df.groupby('Welding_YearMonth').agg(
            焊口总数=('Joint No', 'nunique'),
            参与WPS数=('WPS NO', 'nunique'),
            RT合格率=('RT State', lambda x: (x == '合格').sum() / max(len(x), 1)),
            主要材质=('Material', lambda x: x.mode()[0] if not x.mode().empty else '')
        ).reset_index()
        monthly['Welding_YearMonth'] = monthly['Welding_YearMonth'].astype(str)
        monthly.to_excel(writer, sheet_name='月度趋势', index=False)

        worksheet = writer.sheets['月度趋势']
        chart = workbook.add_chart({'type': 'line'})
        n_rows = len(monthly)
        chart.add_series({'name': '焊口总数', 'categories': f'=月度趋势!$A$2:$A${n_rows+1}', 'values': f'=月度趋势!$B$2:$B${n_rows+1}'})
        chart.add_series({'name': 'RT合格率', 'categories': f'=月度趋势!$A$2:$A${n_rows+1}', 'values': f'=月度趋势!$D$2:$D${n_rows+1}'})
        chart.set_title({'name': '月度焊接趋势'})
        chart.set_x_axis({'name': '月份'})
        chart.set_y_axis({'name': '数量 / 比例'})
        worksheet.insert_chart('G2', chart, {'x_scale': 1.5, 'y_scale': 1.5})

    pipeline = df.groupby('Line No').agg(
        焊口总数=('Joint No', 'nunique'),
        涉及WPS=('WPS NO', lambda x: ', '.join(x.dropna().astype(str).unique())),
        RT合格率=('RT State', lambda x: (x == '合格').sum() / max(len(x), 1)),
        外观合格率=('Visual', lambda x: (x == '合格').sum() / max(len(x), 1))
    ).reset_index()
    pipeline.to_excel(writer, sheet_name='管线质量汇总', index=False)

    root = df.groupby('Root Welder').agg(打底焊口数=('Joint No', 'nunique'), RT合格率打底=('RT State', lambda x: (x == '合格').sum() / max(len(x), 1))).reset_index().rename(columns={'Root Welder': '焊工'})
    cover = df.groupby('Cover Welder').agg(盖面焊口数=('Joint No', 'nunique'), RT合格率盖面=('RT State', lambda x: (x == '合格').sum() / max(len(x), 1))).reset_index().rename(columns={'Cover Welder': '焊工'})
    welder = pd.merge(root, cover, on='焊工', how='outer').fillna(0)
    welder['打底焊口数'] = welder['打底焊口数'].astype(int)
    welder['RT合格率打底'] = welder['RT合格率打底'].astype(float)
    welder['盖面焊口数'] = welder['盖面焊口数'].astype(int)
    welder['RT合格率盖面'] = welder['RT合格率盖面'].astype(float)
    welder.to_excel(writer, sheet_name='焊工质量', index=False)

    r1 = df[df['repair State'].notna() & (df['repair State'] != '')]
    r2 = df[df['Re-repair State'].notna() & (df['Re-repair State'] != '')]
    repair = pd.merge(
        r1.groupby('WPS NO').agg(一返次数=('Joint No', 'count')).reset_index(),
        r2.groupby('WPS NO').agg(二返次数=('Joint No', 'count')).reset_index(),
        on='WPS NO', how='outer')
    repair[['一返次数', '二返次数']] = repair[['一返次数', '二返次数']].fillna(0).astype(int)
    repair.to_excel(writer, sheet_name='返修统计', index=False)

    if 'Test Package No' in df.columns:
        tp = df.groupby(['Test Package No', 'WPS NO']).agg(焊口数=('Joint No', 'nunique')).reset_index()
        tp_pivot = tp.pivot_table(index='Test Package No', columns='WPS NO', values='焊口数', fill_value=0)
        tp_pivot.to_excel(writer, sheet_name='试压包_WPS')

    
    wps_problem = wps_main[wps_main['RT合格率'] < 0.8][['WPS NO', 'RT合格率', '焊口总数']].copy()
    wps_problem['问题类型'] = 'WPS_RT合格率<80%'
    welder_problem = welder[(welder['RT合格率打底'] < 0.8) | (welder['RT合格率盖面'] < 0.8)]
    welder_problem = welder_problem[['焊工', 'RT合格率打底', 'RT合格率盖面', '打底焊口数', '盖面焊口数']].copy()
    welder_problem['问题类型'] = '焊工RT合格率<80%'

    wps_problem.to_excel(writer, sheet_name='问题汇总', startrow=0, index=False)
    welder_problem.to_excel(writer, sheet_name='问题汇总', startrow=len(wps_problem) + 3, index=False)

    red_fmt = workbook.add_format({'bg_color': '#FFC7CE', 'font_color': '#9C0006'})
    worksheet_prob = writer.sheets['问题汇总']
    if len(wps_problem) > 0:
        worksheet_prob.conditional_format(f'B2:B{len(wps_problem)+1}', {'type': 'cell', 'criteria': '<', 'value': 0.8, 'format': red_fmt})
    if len(welder_problem) > 0:
        start_r = len(wps_problem) + 4
        worksheet_prob.conditional_format(f'B{start_r}:C{start_r+len(welder_problem)}', {'type': 'cell', 'criteria': '<', 'value': 0.8, 'format': red_fmt})

print(f"分析完成！报告已生成: {OUTPUT_FILE}")

开始多维分析...
分析完成！报告已生成: D:\lixingjie\WPS_Multidimensional_Report.xlsx
